# Step 6: Odd Finite-Rank Residuals and Physical Norms

Bound the odd inverse trials using a two-sided residual quadratic certificate, an eigenfunction-defect correction, and mass, gradient, and symmetric-part Gram certificates.

Run from `notebooks/` with `dtype = Interval{Float64}`. Inputs are the Step 1 profile, Step 2 normalized eigenpair, Step 4B odd RHS samples and derivative bounds, `data/external/eig.mat`, and the raw trials in `data/external/phi_odd/`.

The results supply the paper's $\Theta_r^v$, $\Theta_\phi^v$, $\Theta_{\nabla\phi}^v$, and $\Theta_{\mathrm s}^v$, appearing in `eq:45-input-rv`, `eq:45-input-Gv`, `eq:45-input-grad-Gv`, and the coercivity estimate `eq:45-v-a`. The complete saved file `step6_odd_matrices_clean.mat` also supplies Step 7.


In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


## Load the fields

Write $\Phi_i$ for the raw loaded inverse velocities, $v$ for the exactly normalized Step 2 eigenfunction, and $P_v=I-v\otimes v$. The manuscript inverse fields are $\overline\phi_i^v=-P_v\Phi_i$, so

$$
G^v=-P_v\Phi\mathcal B^*,\qquad
\mathcal B_i=Q\psi_i/\nu_i.
$$

`eig_val` contains the positive retained generalized eigenvalues $\nu_i$. `lbd` is the corrected unstable eigenvalue, and `lbd_lst` contains the trial constraint multipliers. The corrected field is loaded with its common normalization; it is not normalized again here. Mode counts and coefficient dimensions follow the input arrays.


In [2]:
UP = from_mat_file("../data/generated/UP_divfree.mat", 2; domain="frequency", divfree=true)
up, lbd = from_mat_file("../data/generated/up_eig_corrected.mat", 1; domain="frequency", divfree=true)

rhs_tensor = read_matrix("../data/generated/rhs.mat", ["rhs_odd"])
K = size(rhs_tensor, 3)
R = dt(32)
L0, L1 = Pi / dt(2), Pi / dt(2)
k = 10
padding = derivative_padding(k-2)
Beta = atan(R / UP.scl_fac)

up_phi_lst = []
lbd_lst = []
rhs_lst = []
for ind in 1:K
    println("i = $ind / $K")
    up_phi, lbd_phi = from_mat_file("../data/external/phi_odd/up_phi_$(ind-1).mat", 1; domain="frequency")
    push!(up_phi_lst, up_phi)
    push!(lbd_lst, lbd_phi)
    push!(rhs_lst, VectorFunc(UP.scl_fac, "space";
        e1=rhs_tensor[:,:,ind,1], e2=rhs_tensor[:,:,ind,2], e3=rhs_tensor[:,:,ind,3]))
end
rhs_tensor = nothing
GC.gc(true)

eig_val = read_matrix("../data/external/eig.mat", ["eig_val_odd"])

M0_UP, M1_UP = size(UP["u3"].freq_func)
M0, M1 = size(up_phi_lst[1]["u3"].freq_func)
freq_phi = dt.([2*M0+10, 2*M1+10])
freq_mixed = dt.([2*(M0_UP+M0)+10, 2*(M1_UP+M1)+10])


i = 1 / 25
i = 2 / 25
i = 3 / 25
i = 4 / 25
i = 5 / 25
i = 6 / 25
i = 7 / 25
i = 8 / 25
i = 9 / 25
i = 10 / 25
i = 11 / 25
i = 12 / 25
i = 13 / 25
i = 14 / 25
i = 15 / 25
i = 16 / 25
i = 17 / 25
i = 18 / 25
i = 19 / 25
i = 20 / 25
i = 21 / 25
i = 22 / 25
i = 23 / 25
i = 24 / 25
i = 25 / 25


2-element Vector{Interval{Float64}}:
 [2010.0, 2010.0]_com
 [1010.0, 1010.0]_com

## Derivative bounds

Evaluate the operator, weighted identity, and weighted Laplacian priors on a full β period. Candidate-only and mixed frequency ceilings use the actual coefficient dimensions. The Bessel forcing has explicit radial derivative priors from `rhs_bound.mat`. Residuals and Grams are first computed for the raw trials; the orthogonal projection is applied to their small Gram matrices below.


In [3]:
rhs_norms, lfs_β_norms_prior, err_β_norms_prior, up_Wk, up_Lap_Wk, phi_Wk, phi_Lap_Wk = let
    N0, N1 = 16000, 2000
    L_prior = dt(2)*Pi
    β_pt = grid(L_prior, N0)
    θ_pt = grid(L1, N1)

    lin_mat = get_operator_matrix(up_phi_lst[1], β_pt, θ_pt, "linear")
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
    Lap_mat = get_operator_matrix(up, β_pt, θ_pt, "laplacian")
    lfs_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt))
    add_equal!.(lfs_lst, vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt)))

    rhs_prior_mat = read_matrix("../data/generated/rhs_bound.mat", ["rhs_prior_odd"])
    rhs_β_norms_prior = [rhs_prior_mat[:,:,i,1] for i in 1:k+1]
    lfs_β_norms_prior = [zeros(eltype(M), size(M)) for M in rhs_β_norms_prior]
    rhs_raw = read_matrix("../data/generated/rhs_bound.mat", ["rhs_bound_odd"])
    rhs_norms = [[rhs_raw[:,i,j,l] for i in 1:k-1, j in 1:2] for l in 1:K]

    I_up = apply(I_mat, up)
    up_Wk = norm_lst(I_up, "W$(k-2)Inf"; paras=freq_phi, domain=[L_prior,L1])
    up_Lap_Wk = norm_lst(apply(Lap_mat, up), "W$(k-2)Inf";
        paras=freq_phi, domain=[L_prior,L1])
    phi_Wk, phi_Lap_Wk = [], []

    for ind in 1:K
        println("i = $ind / $K")
        I_phi = apply(I_mat, up_phi_lst[ind])
        push!(phi_Wk, norm_lst(I_phi, "W$(k-2)Inf"; paras=freq_phi, domain=[L_prior,L1]))
        push!(phi_Lap_Wk, norm_lst(apply(Lap_mat, up_phi_lst[ind]), "W$(k-2)Inf";
            paras=freq_phi, domain=[L_prior,L1]))

        lfs = lfs_lst[ind] - apply(lin_mat, up_phi_lst[ind]) - I_phi*lbd - I_up*lbd_lst[ind]
        lfs_norm = norm_lst(lfs, "W$(k)Inf"; paras=freq_mixed, domain=[L_prior,L1])
        for i in 1:k+1, j in 1:3
            lfs_β_norms_prior[i][j,ind] = lfs_norm[i,1][j]
        end
    end
    (rhs_norms, lfs_β_norms_prior, lfs_β_norms_prior .+ rhs_β_norms_prior,
        up_Wk, up_Lap_Wk, phi_Wk, phi_Lap_Wk)
end
GC.gc(true)

i = 1 / 25
i = 2 / 25
i = 3 / 25
i = 4 / 25
i = 5 / 25
i = 6 / 25
i = 7 / 25
i = 8 / 25
i = 9 / 25
i = 10 / 25
i = 11 / 25
i = 12 / 25
i = 13 / 25
i = 14 / 25
i = 15 / 25
i = 16 / 25
i = 17 / 25
i = 18 / 25
i = 19 / 25
i = 20 / 25
i = 21 / 25
i = 22 / 25
i = 23 / 25
i = 24 / 25
i = 25 / 25


## Inner residual and Gram matrices

On $0\le\beta\le\mathrm{Beta}$, the weighted residual is

$$
e_i=w\left[Q\psi_i+(U\cdot\nabla)\varphi_i+(\varphi_i\cdot\nabla)U
-D\varphi_i-\Delta\varphi_i-\nabla p_i-\lambda\varphi_i-\mu_i v\right].
$$

The `rhs_odd` samples retain ten ghost layers until derivative refinement is complete.
After the inner Gram calculation, form the full nonsymmetric cross-Grams G_BR=<B_i,E_j> and G_Bphi=<B_i,phi_j>, together with t_i=<B_i,v>, where B_i=rhs_i/eig_val[i]. Divide only by the row's eigenvalue once. Keep I_up padded until every residual multiplier has been assembled; reuse the trimmed rhs/residual samples and derivative bounds for the cross integrals. Both crosses and t need only this region because the forcing vanishes outside it.


In [4]:
GQ, GR, G_BR, G_Bphi, t = let
    N0, N1 = 12000, 6000
    β_pt = grid(Beta, N0, padding)
    θ_pt = grid(L1, N1, padding)
    lin_mat = get_operator_matrix(up_phi_lst[1], β_pt, θ_pt, "linear")
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
    I_up = apply(I_mat, up)

    err_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt))
    add_equal!.(err_lst, vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt)))
    err_norms = []
    for ind in 1:K
        println("i = $ind / $K")
        lfs = apply(lin_mat, up_phi_lst[ind]) +
            apply(I_mat, up_phi_lst[ind])*lbd + I_up*lbd_lst[ind]
        add_equal!(err_lst[ind], rhs_lst[ind] - lfs)
        paras = Linf_to_paras(err_β_norms_prior, k, [zero(dtype), freq_mixed[2]]; ind=ind)
        push!(err_norms, norm_lst(err_lst[ind], "W$(k-2)Inf";
            paras=paras, domain=[Beta,L1], padding=padding))
        trim_border!(err_lst[ind], padding)
        trim_border!(rhs_lst[ind], padding)
    end

    GQ, GR = zeros(dtype,K,K), zeros(dtype,K,K)
    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        GQ[i,j] = inner_product(rhs_lst[i], rhs_lst[j];
            W_norms_lst=[rhs_norms[i],rhs_norms[j]], domain=[Beta,L1]) / (eig_val[i]*eig_val[j])
        GR[i,j] = inner_product(err_lst[i], err_lst[j];
            W_norms_lst=[err_norms[i],err_norms[j]], domain=[Beta,L1])
        if i != j
            GQ[j,i] = GQ[i,j]
            GR[j,i] = GR[i,j]
        end
    end
    # The forcing is supported on this inner region, so its cross-Grams need no outer term.
    G_BR, G_Bphi = zeros(dtype,K,K), zeros(dtype,K,K)
    t = zeros(dtype,K)
    # Keep padded I_up until all residual multiplier terms above have been assembled.
    trim_border!(I_up, padding)
    for i in 1:K
        t[i] = inner_product(rhs_lst[i], I_up;
            W_norms_lst=[rhs_norms[i],up_Wk], domain=[Beta,L1]) / eig_val[i]
    end
    for j in 1:K
        println("Cross-Gram column $j / $K")
        I_phi = apply(I_mat, up_phi_lst[j])
        trim_border!(I_phi, padding)
        for i in 1:K
            G_BR[i,j] = inner_product(rhs_lst[i], err_lst[j];
                W_norms_lst=[rhs_norms[i],err_norms[j]], domain=[Beta,L1]) / eig_val[i]
            G_Bphi[i,j] = inner_product(rhs_lst[i], I_phi;
                W_norms_lst=[rhs_norms[i],phi_Wk[j]], domain=[Beta,L1]) / eig_val[i]
        end
    end
    GQ, GR, G_BR, G_Bphi, t
end
rhs_lst = nothing
GC.gc(true)

i = 1 / 25
i = 2 / 25
i = 3 / 25
i = 4 / 25
i = 5 / 25
i = 6 / 25
i = 7 / 25
i = 8 / 25
i = 9 / 25
i = 10 / 25
i = 11 / 25
i = 12 / 25
i = 13 / 25
i = 14 / 25
i = 15 / 25
i = 16 / 25
i = 17 / 25
i = 18 / 25
i = 19 / 25
i = 20 / 25
i = 21 / 25
i = 22 / 25
i = 23 / 25
i = 24 / 25
i = 25 / 25
i = 1 / 25, j = 1 / 25
i = 1 / 25, j = 2 / 25
i = 1 / 25, j = 3 / 25
i = 1 / 25, j = 4 / 25
i = 1 / 25, j = 5 / 25
i = 1 / 25, j = 6 / 25
i = 1 / 25, j = 7 / 25
i = 1 / 25, j = 8 / 25
i = 1 / 25, j = 9 / 25
i = 1 / 25, j = 10 / 25
i = 1 / 25, j = 11 / 25
i = 1 / 25, j = 12 / 25
i = 1 / 25, j = 13 / 25
i = 1 / 25, j = 14 / 25
i = 1 / 25, j = 15 / 25
i = 1 / 25, j = 16 / 25
i = 1 / 25, j = 17 / 25
i = 1 / 25, j = 18 / 25
i = 1 / 25, j = 19 / 25
i = 1 / 25, j = 20 / 25
i = 1 / 25, j = 21 / 25
i = 1 / 25, j = 22 / 25
i = 1 / 25, j = 23 / 25
i = 1 / 25, j = 24 / 25
i = 1 / 25, j = 25 / 25
i = 2 / 25, j = 2 / 25
i = 2 / 25, j = 3 / 25
i = 2 / 25, j = 4 / 25
i = 2 / 25, j = 5 / 25
i = 2 / 25, j = 6 / 25
i =

## Outer residual

On $\mathrm{Beta}\le\beta\le\pi/2$, Q and its forcing vanish. Integrate the operator-only residual and add its Gram to the inner contribution, obtaining the whole-space raw `GR` matched to `G_BR`.

The residual bound is checked only after the whole-space projection coefficients b have been computed in the next cell.


In [5]:
let
    N0, N1 = 6000, 6000
    β_pt = grid(L0-Beta, N0, padding) .+ Beta
    θ_pt = grid(L1, N1, padding)
    lin_mat = get_operator_matrix(up_phi_lst[1], β_pt, θ_pt, "linear")
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
    I_up = apply(I_mat, up)

    err_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt))
    add_equal!.(err_lst, vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt)))
    err_norms = []
    for ind in 1:K
        println("i = $ind / $K")
        lfs = apply(lin_mat, up_phi_lst[ind]) +
            apply(I_mat, up_phi_lst[ind])*lbd + I_up*lbd_lst[ind]
        minus_equal!(err_lst[ind], lfs)
        paras = Linf_to_paras(lfs_β_norms_prior, k, [zero(dtype), freq_mixed[2]]; ind=ind)
        push!(err_norms, norm_lst(err_lst[ind], "W$(k-2)Inf";
            paras=paras, domain=[L0-Beta,L1], padding=padding))
        trim_border!(err_lst[ind], padding)
    end

    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        GR[i,j] += inner_product(err_lst[i], err_lst[j];
            W_norms_lst=[err_norms[i],err_norms[j]], domain=[L0-Beta,L1])
        if i != j
            GR[j,i] = GR[i,j]
        end
    end
end
GC.gc(true)

i = 1 / 25
i = 2 / 25
i = 3 / 25
i = 4 / 25
i = 5 / 25
i = 6 / 25
i = 7 / 25
i = 8 / 25
i = 9 / 25
i = 10 / 25
i = 11 / 25
i = 12 / 25
i = 13 / 25
i = 14 / 25
i = 15 / 25
i = 16 / 25
i = 17 / 25
i = 18 / 25
i = 19 / 25
i = 20 / 25
i = 21 / 25
i = 22 / 25
i = 23 / 25
i = 24 / 25
i = 25 / 25
i = 1 / 25, j = 1 / 25
i = 1 / 25, j = 2 / 25
i = 1 / 25, j = 3 / 25
i = 1 / 25, j = 4 / 25
i = 1 / 25, j = 5 / 25
i = 1 / 25, j = 6 / 25
i = 1 / 25, j = 7 / 25
i = 1 / 25, j = 8 / 25
i = 1 / 25, j = 9 / 25
i = 1 / 25, j = 10 / 25
i = 1 / 25, j = 11 / 25
i = 1 / 25, j = 12 / 25
i = 1 / 25, j = 13 / 25
i = 1 / 25, j = 14 / 25
i = 1 / 25, j = 15 / 25
i = 1 / 25, j = 16 / 25
i = 1 / 25, j = 17 / 25
i = 1 / 25, j = 18 / 25
i = 1 / 25, j = 19 / 25
i = 1 / 25, j = 20 / 25
i = 1 / 25, j = 21 / 25
i = 1 / 25, j = 22 / 25
i = 1 / 25, j = 23 / 25
i = 1 / 25, j = 24 / 25
i = 1 / 25, j = 25 / 25
i = 2 / 25, j = 2 / 25
i = 2 / 25, j = 3 / 25
i = 2 / 25, j = 4 / 25
i = 2 / 25, j = 5 / 25
i = 2 / 25, j = 6 / 25
i =

## Whole-space Grams and spectral bounds

For raw trials $\Phi_i$, define

$$
H_{ij}=\langle\Phi_i,\Phi_j\rangle,\qquad
D_{ij}=-\langle\Phi_i,\Delta\Phi_j\rangle,
$$

$$
b_i=\langle v,\Phi_i\rangle,\qquad
d_i=-\langle v,\Delta\Phi_i\rangle,\qquad
d_0=-\langle v,\Delta v\rangle.
$$

These Laplacian integrals equal the corresponding physical gradient inner products by integration by parts.

The raw residual block Gram `[GQ G_BR; G_BR' GR]` certifies both signs of its quadratic form. The remaining budget is

$$
\texttt{K7\_v}-\texttt{K7\_v\_correction},\qquad
\texttt{K7\_v}=0.02690,\qquad
\texttt{K7\_v\_correction}=4.808\times10^{-6}\sqrt{|b^*GQb|}.
$$

The correction uses Step 2's strong-residual bound for the same normalized eigenpair. The exact Gram quadratic form is nonnegative; the absolute value handles an interval enclosure crossing zero. This gives the paper's $\Theta_r^v\le0.027$.

Project the small matrices by $H_\perp=H-bb^*$ and $X_\perp=G_{B\phi}-tb^*$. Their Gram certificates check `A0_v=52.01` and `B0_v=52.20`. The gradient projection below checks `A1_v=45.51`. Thus the paper's rounded inputs $\Theta_\phi^v\le52.1$ and $\Theta_{\nabla\phi}^v\le45.6$ follow.

**Symmetric part.** The two calls with `K_quad` and `-K_quad` both use threshold `32.52`. They give

$$
\left\|\frac{G^v+(G^v)^*}{2}\right\|\le32.52,
\qquad
\Theta_{\mathrm s}^v:=1+\left\|\frac{G^v+(G^v)^*}{2}\right\|
\le33.52<33.6.
$$

The saved name `Bquad_v` denotes this $33.52$ upper bound. It also bounds $\|\operatorname{sym}(I+G^v)\|$ by the triangle inequality; the paper defines $\Theta_{\mathrm s}^v$ with the identity outside the norm.

`verify_gram_spectrum` constructs matrix upper bounds U and V and checks $U\succ0$ and $tU-UVU\succ0$ with interval LDL. The exact input is a Gram matrix and may be singular. See [the certificate derivation](../docs/technical_notes/matrix_func.md#symmetric-and-gram-spectral-certificates).


In [ ]:
(G_phi, G_Dphi, G_phi_proj, G_Dphi_proj, G_Bphi_proj, b, d, d0,
    K7_v_correction, K7_v, Bquad_v) = let
    N0, N1 = 12000, 6000
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)
    I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
    Lap_mat = get_operator_matrix(up, β_pt, θ_pt, "laplacian")
    I_up = apply(I_mat, up)

    I_phi_lst = []
    b, d = zeros(dtype,K), zeros(dtype,K)
    for i in 1:K
        println("i = $i / $K")
        I_phi = apply(I_mat, up_phi_lst[i])
        push!(I_phi_lst, I_phi)
        b[i] = inner_product(I_up, I_phi;
            W_norms_lst=[up_Wk,phi_Wk[i]], domain=[L0,L1])
        println("Projection coefficient b[", i, "]: ", b[i])
    end

    K7_v_correction = dt("4.808e-6") * sqrt(abs(b' * GQ * b))
    println("K7_v projection correction: ", K7_v_correction)

    I_K = one(GQ)
    Z = zeros(dtype,K,K)
    W_r = [GQ G_BR; G_BR' GR]
    K_r = [Z -I_K/dt(2); -I_K/dt(2) Z]
    K7_v = dt("0.02690")
    println("K7_v bound: ", K7_v)
    @assert verify_gram_spectrum(W_r, K_r, K7_v-K7_v_correction)
    @assert verify_gram_spectrum(W_r, -K_r, K7_v-K7_v_correction)

    G_phi, G_Dphi = zeros(dtype,K,K), zeros(dtype,K,K)
    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        G_phi[i,j] = inner_product(I_phi_lst[i], I_phi_lst[j];
            W_norms_lst=[phi_Wk[i],phi_Wk[j]], domain=[L0,L1])
        if i != j
            G_phi[j,i] = G_phi[i,j]
        end
    end

    G_phi_proj = G_phi - b*b'
    G_Bphi_proj = G_Bphi - t*b'

    println("A0_v bound: ", dt("52.01"))
    @assert verify_gram_spectrum(GQ, G_phi_proj, dt("52.01")^2)

    W_phi = [GQ G_Bphi_proj; G_Bphi_proj' G_phi_proj]
    K_norm = [G_phi_proj -I_K; -I_K Z]
    println("B0_v bound: ", dt("52.20"))
    @assert verify_gram_spectrum(W_phi, K_norm, dt("52.20")^2-one(dtype))

    K_quad = [Z -I_K/dt(2); -I_K/dt(2) Z]
    Bquad_v = dt("33.52")
    println("Bquad_v bound: ", Bquad_v)
    @assert verify_gram_spectrum(W_phi, K_quad, dt("32.52"))
    @assert verify_gram_spectrum(W_phi, -K_quad, dt("32.52"))

    d0 = -inner_product(I_up, apply(Lap_mat, up);
        W_norms_lst=[up_Wk,up_Lap_Wk], domain=[L0,L1])
    for j in 1:K
        println("Laplacian $j / $K")
        Lap_phi = apply(Lap_mat, up_phi_lst[j])
        d[j] = -inner_product(I_up, Lap_phi;
            W_norms_lst=[up_Wk,phi_Lap_Wk[j]], domain=[L0,L1])
        for i in 1:j
            G_Dphi[i,j] = -inner_product(I_phi_lst[i], Lap_phi;
                W_norms_lst=[phi_Wk[i],phi_Lap_Wk[j]], domain=[L0,L1])
            if i != j
                G_Dphi[j,i] = G_Dphi[i,j]
            end
        end
    end
    G_Dphi_proj = [G_Dphi[i,j]-b[i]*d[j]-d[i]*b[j]+d0*b[i]*b[j]
        for i in 1:K, j in 1:K]
    println("A1_v bound: ", dt("45.51"))
    @assert verify_gram_spectrum(GQ, G_Dphi_proj, dt("45.51")^2)

    (G_phi, G_Dphi, G_phi_proj, G_Dphi_proj, G_Bphi_proj, b, d, d0,
        K7_v_correction, K7_v, Bquad_v)
end
GC.gc(true)

## Late projection and saved bounds

For unit v, the projected mass, gradient, and forcing/trial cross matrices are

$$
H_\perp=H-bb^*,\qquad
D_\perp=D-bd^*-db^*+d_0bb^*,\qquad
X_\perp=X-tb^*.
$$

The negative off-diagonal blocks implement $G^v=-P_v\Phi\mathcal B^*$. The raw residual block is used with the single eigenfunction-defect allowance above; pressure gradients and multiples of v pair to zero with the divergence-free, v-orthogonal test fields.

The final `step6_odd_matrices_clean.mat` contains raw `GQ,GR,G_BR,G_Bphi,G_phi,G_Dphi`; projected `G_Bphi_proj,G_phi_proj,G_Dphi_proj`; `b,d,d0,t`; and `K7_v_correction,K7_v,Bquad_v`. The inline A0, A1, and B0 thresholds are not saved.

[Step 7](verify_auxiliary_field_bound.ipynb) reads `GQ,GR,t,b` from this complete final file. Its inverse multipliers, normalized v, retained basis, and Step 4 RHS must match those used here. The inner checkpoint lacks the exterior residual contribution and is not the Step 7 input.


In [7]:
# GR remains the complete raw-residual Gram; the projection correction is counted in K7_v.
save_matrix("../data/generated/step6_odd_matrices_clean.mat", Dict(
    "GQ" => GQ, "GR" => GR,
    "G_BR" => G_BR, "G_Bphi" => G_Bphi, "G_Bphi_proj" => G_Bphi_proj,
    "G_phi" => G_phi, "G_Dphi" => G_Dphi,
    "G_phi_proj" => G_phi_proj, "G_Dphi_proj" => G_Dphi_proj,
    "b" => reshape(b,K,1), "d" => reshape(d,K,1), "d0" => reshape([d0],1,1),
    "t" => reshape(t,K,1),
    "K7_v_correction" => K7_v_correction, "K7_v" => K7_v,
    "Bquad_v" => Bquad_v,
))
